# Create 7 Disease-Specific Datasets for Single-Label Prediction

Pipeline:
1. Assign each patient to ONE disease (earliest fibrotic visit)
2. Filter case patients (must have prior EHR history)
3. Identify fibrotic ICD columns to remove
4. Build datasets: Cases (y=1) + Controls (y=0) per disease
5. Validate outputs

## Section 0: Config & Imports

In [ ]:
import pandas as pd
import os
import gc
import warnings
warnings.filterwarnings('ignore')

# ── Toggle between sample and full data ──
USE_SAMPLE = False

if USE_SAMPLE:
    FIBROTIC_VISIT_FILE = "fibrotic_gpX_hesinY_Y_visit_sample.csv"
    RAW_EHR_FILE = "gp_icd10_hesin_raw_data_with_ukb_sample.csv"
else:
    FIBROTIC_VISIT_FILE = "fibrotic_gpX_hesinY_Y_visit.csv"
    RAW_EHR_FILE = "gp_icd10_hesin_raw_data_with_ukb.csv"

OUTPUT_DIR = "datasets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Chunked reading config ──
CHUNK_SIZE = 50_000   # rows per chunk for reading raw EHR
CTRL_BATCH = 500      # flush controls to CSV every N patients

# ── 7 diseases ──
DISEASES = [
    "CKD", "Cardiac_Fibrosis", "MASH", "Pulmonary_fibrosis",
    "SSc_Connective_Tissue", "Crohns_Disease", "Fibrosis_of_Skin"
]

# ── Fibrotic ICD codes per disease (column format: dots removed) ──
FIBROTIC_ICD_CODES = {
    "CKD":                   ["N018", "N181", "N182", "N183", "N184", "N185", "N189"],
    "Cardiac_Fibrosis":      ["I040", "I409", "I420", "I423", "I424", "I425", "I514"],
    "MASH":                  ["K074", "K746", "K75", "K758", "K7581"],
    "Pulmonary_fibrosis":    ["J841", "J8410", "J84112", "J8417", "J848", "J8489", "J849"],
    "SSc_Connective_Tissue": ["M034", "M340", "M341", "M342", "M348", "M349"],
    "Crohns_Disease":        ["K050", "K501", "K508", "K509"],
    "Fibrosis_of_Skin":      ["L905", "L091", "L910"],
}

# Flatten to a single set of all fibrotic codes to remove
ALL_FIBROTIC_CODES = set()
for codes in FIBROTIC_ICD_CODES.values():
    ALL_FIBROTIC_CODES.update(codes)

print(f"Total fibrotic ICD codes to remove: {len(ALL_FIBROTIC_CODES)}")
print(f"Using {'sample' if USE_SAMPLE else 'full'} data")
print(f"Chunk size: {CHUNK_SIZE:,} rows")
print(f"Fibrotic visit file: {FIBROTIC_VISIT_FILE}")
print(f"Raw EHR file: {RAW_EHR_FILE}")
print(f"Output directory: {OUTPUT_DIR}/")

## Step 1: Assign Each Patient to ONE Disease

For each patient, find the **earliest** visit in the fibrotic visit file and assign the disease that has a flag of 1. If multiple diseases = 1 at the same earliest visit, **skip** that patient.

In [ ]:
# Load fibrotic visit data
visit_df = pd.read_csv(FIBROTIC_VISIT_FILE, parse_dates=["event_dt"])
print(f"Fibrotic visit file: {len(visit_df)} rows, {visit_df['eid'].nunique()} unique patients")
visit_df.head(10)

In [ ]:
# For each patient, get the earliest visit row
visit_df = visit_df.sort_values(["eid", "event_dt"])
earliest_visits = visit_df.groupby("eid").first().reset_index()

# Identify which disease(s) = 1 at the earliest visit
patient_assignment = {}  # eid -> (disease_name, diagnosis_date)
skipped_multi = []       # patients with multiple diseases at earliest visit

for _, row in earliest_visits.iterrows():
    eid = row["eid"]
    diag_date = row["event_dt"]
    active_diseases = [d for d in DISEASES if row[d] == 1]
    
    if len(active_diseases) == 0:
        # This shouldn't happen (every row in fibrotic visit file should have at least one disease=1)
        print(f"  WARNING: eid={eid} has no disease=1 at earliest visit {diag_date}")
    elif len(active_diseases) > 1:
        # Multiple diseases at the same earliest visit -> skip
        skipped_multi.append(eid)
        print(f"  WARNING: eid={eid} has multiple diseases at earliest visit {diag_date}: {active_diseases} -> SKIPPED")
    else:
        patient_assignment[eid] = (active_diseases[0], diag_date)

print(f"\nPatients assigned to a disease: {len(patient_assignment)}")
print(f"Patients skipped (multi-disease): {len(skipped_multi)}")

# Show assignment counts per disease
from collections import Counter
disease_counts = Counter(v[0] for v in patient_assignment.values())
print("\nAssignment counts per disease:")
for d in DISEASES:
    print(f"  {d}: {disease_counts.get(d, 0)}")

## Step 2: Prior History Filter (Case patients only)

For each case patient, check if their diagnosis date is their **first-ever** visit in the raw EHR data. If yes, exclude them (no prior history for X features). Controls are NOT filtered.

In [ ]:
# ── Read only eid + event_dt in CHUNKS to find each patient's earliest visit ──
CHUNK_SIZE = 100000  # rows per chunk — adjust based on your machine's RAM

earliest_raw = {}  # eid -> earliest event_dt
total_rows = 0

for chunk in pd.read_csv(RAW_EHR_FILE, usecols=["eid", "event_dt"],
                          parse_dates=["event_dt"], chunksize=CHUNK_SIZE):
    total_rows += len(chunk)
    chunk_min = chunk.groupby("eid")["event_dt"].min()
    for eid, dt in chunk_min.items():
        if eid not in earliest_raw or dt < earliest_raw[eid]:
            earliest_raw[eid] = dt
    # progress
    print(f"  processed {total_rows:,} rows ...", end="\r")

print(f"\nRaw EHR data: {total_rows:,} total rows, {len(earliest_raw):,} unique patients")

# ── Filter case patients ──
case_patients = {}       # eid -> (disease_name, diagnosis_date)  [filtered]
excluded_no_history = []
excluded_not_in_raw = []

for eid, (disease, diag_date) in patient_assignment.items():
    if eid not in earliest_raw:
        excluded_not_in_raw.append(eid)
        continue
    
    patient_earliest = earliest_raw[eid]
    if diag_date <= patient_earliest:
        excluded_no_history.append(eid)
    else:
        case_patients[eid] = (disease, diag_date)

print(f"\nCase patients after filtering: {len(case_patients)}")
print(f"Excluded (no prior history): {len(excluded_no_history)}")
print(f"Excluded (not in raw EHR): {len(excluded_not_in_raw)}")

# Show filtered counts per disease
filtered_counts = Counter(v[0] for v in case_patients.values())
print("\nFiltered case counts per disease:")
for d in DISEASES:
    before = disease_counts.get(d, 0)
    after = filtered_counts.get(d, 0)
    print(f"  {d}: {before} -> {after} ({before - after} excluded)")

## Step 3: Identify Fibrotic ICD Columns to Remove

Match the 39 fibrotic ICD codes against raw EHR column names using **exact string matching**. Non-fibrotic codes that share a prefix (e.g., I421, K743) are kept.

In [ ]:
# Read only the header of the raw EHR file
raw_header = pd.read_csv(RAW_EHR_FILE, nrows=0).columns.tolist()
print(f"Total columns in raw EHR: {len(raw_header)}")

# Find which fibrotic codes exist as columns (exact match)
fibrotic_cols_found = ALL_FIBROTIC_CODES.intersection(set(raw_header))
fibrotic_cols_not_found = ALL_FIBROTIC_CODES - fibrotic_cols_found

print(f"\nFibrotic ICD codes found in raw EHR columns ({len(fibrotic_cols_found)}):")
print(f"  {sorted(fibrotic_cols_found)}")
print(f"\nFibrotic ICD codes NOT found ({len(fibrotic_cols_not_found)}):")
print(f"  {sorted(fibrotic_cols_not_found)}")

# Columns to keep: everything except fibrotic codes
cols_to_drop = list(fibrotic_cols_found)
cols_to_keep = [c for c in raw_header if c not in fibrotic_cols_found]
print(f"\nColumns to keep: {len(cols_to_keep)} (dropped {len(cols_to_drop)} fibrotic columns)")

## Step 4: Build 7 Disease Datasets (Chunked Processing)

**Memory-safe approach:**
1. Collect all relevant eids (case + control = all assigned patients)
2. Read raw EHR in chunks, keeping **only** rows for relevant eids (a small fraction of total rows)
3. After accumulation, build and write each disease's CSV incrementally

In [ ]:
# ── Pass 1: Collect all relevant eids ──
# Case patients: passed Step 2 filter
# Control patients: all assigned patients who are NOT the target disease (varies per disease,
#   but the union = all assigned patients). So we need ALL assigned patients' data.
all_relevant_eids = set(patient_assignment.keys())  # every assigned patient
print(f"Relevant eids to extract: {len(all_relevant_eids):,}")
print(f"  (case patients who passed filter: {len(case_patients):,})")

# ── Pass 2: Read raw EHR in chunks, keep only relevant patients ──
import gc

filtered_chunks = []
total_rows = 0
kept_rows = 0

for chunk in pd.read_csv(RAW_EHR_FILE, usecols=cols_to_keep,
                          parse_dates=["event_dt"], chunksize=CHUNK_SIZE):
    total_rows += len(chunk)
    
    # Keep only rows for relevant patients
    mask = chunk["eid"].isin(all_relevant_eids)
    filtered = chunk[mask]
    
    if len(filtered) > 0:
        filtered_chunks.append(filtered)
        kept_rows += len(filtered)
    
    del chunk, mask, filtered
    gc.collect()
    print(f"  processed {total_rows:,} rows, kept {kept_rows:,} ...", end="\r")

# ── Combine filtered data ──
if filtered_chunks:
    raw_ehr = pd.concat(filtered_chunks, ignore_index=True)
    del filtered_chunks
    gc.collect()
else:
    raw_ehr = pd.DataFrame(columns=cols_to_keep)

print(f"\nDone! Total rows scanned: {total_rows:,}")
print(f"Rows kept (relevant patients): {len(raw_ehr):,}  ({len(raw_ehr)/max(total_rows,1)*100:.1f}%)")
print(f"Unique patients in filtered data: {raw_ehr['eid'].nunique():,}")
print(f"Columns: {len(raw_ehr.columns)}")
print(f"Memory usage: {raw_ehr.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

In [ ]:
# ── Build & write each disease dataset one at a time ──
dataset_stats = []
all_assigned = patient_assignment  # eid -> (disease, diag_date)

for disease in DISEASES:
    print(f"\n{'='*60}")
    print(f"Building dataset for: {disease}")
    print(f"{'='*60}")
    
    # ── Cases (y=1): patients assigned to this disease who passed Step 2 ──
    case_eids = {eid for eid, (d, _) in case_patients.items() if d == disease}
    
    # ── Controls (y=0): patients assigned to ANY OTHER disease ──
    control_eids = {eid for eid, (d, _) in all_assigned.items() if d != disease}
    
    print(f"  Case patients: {len(case_eids)}")
    print(f"  Control patients: {len(control_eids)}")
    
    output_path = os.path.join(OUTPUT_DIR, f"dataset_{disease}.csv")
    header_written = False
    case_eids_with_data = 0
    control_eids_with_data = 0
    n_x, n_y, n_ctrl = 0, 0, 0
    
    # ── Column order ──
    meta_cols = ["eid", "event_dt", "y_label", "record_type"]
    feature_cols = [c for c in raw_ehr.columns if c not in meta_cols and c != "eid" and c != "event_dt"]
    col_order = meta_cols + feature_cols
    
    # ── Process case patients: write in small batches ──
    case_batch = []
    for eid in case_eids:
        patient_rows = raw_ehr.loc[raw_ehr["eid"] == eid].copy()
        if len(patient_rows) == 0:
            continue
        
        case_eids_with_data += 1
        diag_date = case_patients[eid][1]
        
        # X rows: visits BEFORE diagnosis date
        x_rows = patient_rows[patient_rows["event_dt"] < diag_date].copy()
        x_rows["y_label"] = 1
        x_rows["record_type"] = "x_row"
        
        # Y row: visit ON diagnosis date
        y_rows = patient_rows[patient_rows["event_dt"] == diag_date].copy()
        y_rows["y_label"] = 1
        y_rows["record_type"] = "y_row"
        
        case_batch.append(x_rows)
        case_batch.append(y_rows)
        n_x += len(x_rows)
        n_y += len(y_rows)
    
    # Write case rows
    if case_batch:
        case_df = pd.concat(case_batch, ignore_index=True)[col_order]
        case_df.to_csv(output_path, index=False, mode="w")
        header_written = True
        del case_df, case_batch
        gc.collect()
    
    # ── Process control patients: write in batches to limit memory ──
    CTRL_BATCH = 500  # flush every N control patients
    ctrl_batch = []
    ctrl_count = 0
    
    for eid in control_eids:
        patient_rows = raw_ehr.loc[raw_ehr["eid"] == eid].copy()
        if len(patient_rows) == 0:
            continue
        
        control_eids_with_data += 1
        patient_rows["y_label"] = 0
        patient_rows["record_type"] = "control"
        ctrl_batch.append(patient_rows)
        n_ctrl += len(patient_rows)
        ctrl_count += 1
        
        # Flush batch to disk
        if ctrl_count % CTRL_BATCH == 0:
            batch_df = pd.concat(ctrl_batch, ignore_index=True)[col_order]
            batch_df.to_csv(output_path, index=False, mode="a",
                           header=(not header_written))
            header_written = True
            del batch_df
            ctrl_batch = []
            gc.collect()
    
    # Flush remaining controls
    if ctrl_batch:
        batch_df = pd.concat(ctrl_batch, ignore_index=True)[col_order]
        batch_df.to_csv(output_path, index=False, mode="a",
                       header=(not header_written))
        header_written = True
        del batch_df, ctrl_batch
        gc.collect()
    
    total_rows_out = n_x + n_y + n_ctrl
    print(f"  Cases with data in raw EHR: {case_eids_with_data}")
    print(f"  Controls with data in raw EHR: {control_eids_with_data}")
    print(f"  Output rows: {total_rows_out:,} (x_row={n_x:,}, y_row={n_y:,}, control={n_ctrl:,})")
    
    if total_rows_out > 0:
        print(f"  Saved to: {output_path}")
    else:
        # Remove empty file if nothing was written
        if os.path.exists(output_path):
            os.remove(output_path)
        print(f"  No data found for this disease")
    
    dataset_stats.append({
        "disease": disease,
        "n_cases": case_eids_with_data,
        "n_controls": control_eids_with_data,
        "n_x_rows": n_x,
        "n_y_rows": n_y,
        "n_control_rows": n_ctrl,
        "n_total_rows": total_rows_out,
    })

print(f"\n{'='*60}")
print("All datasets built!")
print(f"{'='*60}")

## Step 5: Validation & Summary

Verify:
1. No fibrotic ICD columns leaked into output
2. No patient appears as both case and control in the same dataset
3. Every case patient has at least 1 x_row and exactly 1 y_row (when data exists)

In [ ]:
# ── Summary table ──
stats_df = pd.DataFrame(dataset_stats)
print("=" * 80)
print("DATASET SUMMARY")
print("=" * 80)
print(stats_df.to_string(index=False))
print()

# ── Validation ──
all_passed = True

for disease in DISEASES:
    output_path = os.path.join(OUTPUT_DIR, f"dataset_{disease}.csv")
    if not os.path.exists(output_path):
        print(f"[SKIP] {disease}: no output file (likely no matching patients in sample data)")
        continue
    
    df = pd.read_csv(output_path)
    
    # Check 1: No fibrotic columns
    leaked = set(df.columns) & fibrotic_cols_found
    if leaked:
        print(f"[FAIL] {disease}: fibrotic columns leaked: {leaked}")
        all_passed = False
    else:
        print(f"[PASS] {disease}: no fibrotic columns in output")
    
    # Check 2: No case/control overlap
    case_eids_out = set(df[df["y_label"] == 1]["eid"].unique())
    ctrl_eids_out = set(df[df["y_label"] == 0]["eid"].unique())
    overlap = case_eids_out & ctrl_eids_out
    if overlap:
        print(f"[FAIL] {disease}: {len(overlap)} patients appear as both case and control")
        all_passed = False
    else:
        print(f"[PASS] {disease}: no case/control overlap")
    
    # Check 3: Case patients have x_rows and y_rows
    if len(case_eids_out) > 0:
        for eid in case_eids_out:
            patient_data = df[df["eid"] == eid]
            n_x = len(patient_data[patient_data["record_type"] == "x_row"])
            n_y = len(patient_data[patient_data["record_type"] == "y_row"])
            if n_x == 0:
                print(f"[WARN] {disease}: case eid={eid} has 0 x_rows (no prior visits in raw EHR)")
            if n_y == 0:
                print(f"[WARN] {disease}: case eid={eid} has 0 y_rows (diagnosis visit not found in raw EHR)")

print()
if all_passed:
    print("All validations PASSED!")
else:
    print("Some validations FAILED — check above for details.")